# Case Study 3: Sentiment Analysis on Amazon Product Reviews - Dataset Loading

**Labels:** Negative (1-2 stars), Neutral (3 stars), Positive (4-5 stars).
**Dataset:** Amazon Reviews 2023 (McAuley Lab), *Health and Personal Care* category, streamed from the raw `.jsonl` file (balanced sampling).


## 1. Install dependencies


In [ ]:
!pip -q install datasets

## 2. Imports and configuration


In [2]:
import os, json, random, warnings
import numpy as np, pandas as pd
import requests
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# ---------------- CONFIG ----------------
SEED        = 42
PER_CLASS   = 2000            # 3 x 2000 = 6000 balanced reviews
LOCAL_CSV   = None            # e.g. "my_reviews.csv" with columns: text, rating (skips download)
LABEL_NAMES = ["Negative", "Neutral", "Positive"]
# ----------------------------------------

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)

set_all_seeds(SEED)


## 3. Load Amazon reviews (streaming, balanced)
Stream the raw `.jsonl` file line by line and stop as soon as we have `PER_CLASS` reviews for each class. The result is cached to `amazon_subset.csv` so re-runs are instant.


In [3]:
URL = ("https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/"
       "raw/review_categories/Health_and_Personal_Care.jsonl")
CACHE_FILE = "amazon_subset.csv"

def rating_to_label(r):
    # 1-2 -> Negative (0), 3 -> Neutral (1), 4-5 -> Positive (2)
    return 0 if r <= 2 else (1 if r == 3 else 2)

def stream_balanced_reviews(url, per_class, max_scan=800_000):
    buckets = {0: [], 1: [], 2: []}
    seen = set()
    with requests.get(url, stream=True, timeout=60) as resp:
        resp.raise_for_status()
        resp.encoding = "utf-8"
        for i, line in enumerate(resp.iter_lines(decode_unicode=True)):
            if i >= max_scan:
                break
            if not line:
                continue
            rec = json.loads(line)
            if rec.get("rating") is None:
                continue
            rating = int(round(float(rec["rating"])))
            if rating not in (1, 2, 3, 4, 5):
                continue
            title = (rec.get("title") or "").strip()
            body = (rec.get("text") or "").strip()
            if len(body) < 10:
                continue
            text = f"{title}. {body}" if title else body      # title carries strong sentiment signal
            if text in seen:
                continue
            lab = rating_to_label(rating)
            if len(buckets[lab]) < per_class:
                buckets[lab].append(text)
                seen.add(text)
            if all(len(v) >= per_class for v in buckets.values()):
                break
    return buckets

if LOCAL_CSV:
    df = pd.read_csv(LOCAL_CSV)[["text", "rating"]].dropna()
    df["label"] = df["rating"].astype(int).apply(rating_to_label)
    df = df[["text", "label"]]
elif os.path.exists(CACHE_FILE):
    df = pd.read_csv(CACHE_FILE)
    print("Loaded cached subset:", CACHE_FILE)
else:
    try:
        buckets = stream_balanced_reviews(URL, PER_CLASS)
    except Exception as e:
        raise RuntimeError(
            f"Streaming download failed: {e!r}\n"
            "Fallback: download any Amazon reviews CSV with columns 'text' and 'rating', "
            "upload it to Colab and set LOCAL_CSV at the top.")
    rows = [(t, lab) for lab, texts in buckets.items() for t in texts]
    df = pd.DataFrame(rows, columns=["text", "label"])
    df.to_csv(CACHE_FILE, index=False)

df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(df.shape)
print(df["label"].map(dict(enumerate(LABEL_NAMES))).value_counts())
if df["label"].value_counts().min() < PER_CLASS:
    print("WARNING: one class has fewer than PER_CLASS samples (the file ended early). Results are still valid but less balanced.")
df.head()


(6000, 2)
label
Negative    2000
Neutral     2000
Positive    2000
Name: count, dtype: int64


,text,label
0,Not good. Upset stomach,0
1,The company did give good support. I believe t...,1
2,Didn’t live up to expectations.. We bought thi...,0
3,This is too big to travel with and hide when n...,1
4,Great addition to my healthy bean brownies! Re...,2


## 4. Train / validation / test split (80 / 10 / 10, stratified)


In [4]:
train_df, temp_df = train_test_split(df, test_size=0.20, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)
train_df, val_df, test_df = [x.reset_index(drop=True) for x in (train_df, val_df, test_df)]

data = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df),
})
print(data)


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 4800
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 600
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 600
    })
})
